In [ ]:
import ee, eemont
import geemap
import geemap.colormaps as geecm

In [ ]:
ee.Authenticate()
ee.Initialize()

In [ ]:
from pygee.tools.lc_mapping import LandCoverMap

In [ ]:
pygee_dir = "to_complete_with_pygee_install_dir/"

In [ ]:
mapping_file = os.path.join(pygee_dir, "tools/nomenclature_h1b.txt"
mapping_name = "h1b_paper"

In [ ]:
lcmap = LandCoverMap(mapping_file, name=mapping_name)

# 1. Merging

In [ ]:
carto_g25 = ee.Image("users/aguerou/ice_and_life/carto_h1b/carto/carto_h1b_lia_Reinthaler_Mourey_median_ts_g25_rf_only")
water_classif = ee.Image("users/aguerou/ice_and_life/carto_h1b/carto_sensitivity/water_lia_Reinthaler_Mourey_2025_withbuffer200_ndwi015_slope20")

In [ ]:
water_val = int(lcmap.df.loc[lcmap.df["Type"]=="water","Code"].iloc[0])

In [ ]:
carto_g25_with_water = carto_g25.where(water_classif.select("water")==water_val, water_val)

## 1.1 Set glacier value within RGI

In [ ]:
fc_rgi = ee.FeatureCollection("users/aguerou/ice_and_life/carto_h1b/lia_shp/c3s_gi_rgi11_s2_2015_v2_within_lia_reinthaler24_mourey25")

In [ ]:
rgi_image = fc_rgi.reduceToImage(properties=['index'], reducer=ee.Reducer.first())

In [ ]:
carto_g25_with_water_ice = carto_g25_with_water.where(rgi_image.mask()==1, int(lcmap.df.loc[lcmap.df["Type"]=="snow & ice","Code"].iloc[0]))

## 1.2 Visualisation

In [ ]:
Map = geemap.Map(basemap='Esri.WorldImagery')
Map.centerObject(carto_g25_with_water_ice, 8)
Map.addLayer(carto_g25_with_water_ice.updateMask(carto_g25_with_water_ice.gt(-1)), {"min":-1, "max":6, "palette":["#ffffff"]+list(lcmap.get_colors()[:-2])})
Map

# 2. Savings

In [ ]:
fc_lia = ee.FeatureCollection("users/aguerou/ice_and_life/carto_h1b/lia_shp/LIA_Alps_Reinthaler_Paul_2024_Mourey_2025_with_buffer200")

In [ ]:
scale = 20
projection = "EPSG:3035"

## Assets

Takes about 10 min

In [ ]:
geemap.ee_export_image_to_asset(carto_g25_with_water_ice.updateMask(carto_g25_with_water_ice.gt(-1)).clip(fc_lia),
                                assetId="users/aguerou/ice_and_life/carto_h1b/carto/carto_h1b_lia_Reinthaler_Mourey_median_ts_g25",
                                scale=scale,
                                crs=projection,
                                maxPixels=3e10,
                                region=fc_lia.geometry().convexHull())

## Drive

Takes about 10 min

In [ ]:
geemap.ee_export_image_to_drive(carto_g25_with_water_ice, #pas de clip pour que le nodata marche
                                folder="g25_final", 
                                fileNamePrefix="carto_h1b_lia_Reinthaler_Mourey_median_ts_g25", 
                                scale=scale,
                                crs=projection,
                                maxPixels=3e10,
                                region=fc_lia.geometry().convexHull(),
                                formatOptions={"noData": -1})

No buffer - to be able to dissociate

In [ ]:
fc_lia_no_buff = ee.FeatureCollection("users/aguerou/ice_and_life/carto_h1b/lia_shp/LIA_Alps_Reinthaler_Paul_2024_Mourey_2025")

In [ ]:
geemap.ee_export_image_to_drive(carto_g25_with_water_ice.clip(fc_lia_no_buff).unmask(value=-1), #pas de clip pour que le nodata marche
                                folder="g25_final", 
                                fileNamePrefix="carto_h1b_lia_Reinthaler_Mourey_median_ts_g25_no_buffer", 
                                scale=scale,
                                crs=projection,
                                maxPixels=3e10,
                                region=fc_lia.geometry().convexHull(),
                                formatOptions={"noData": -1})

Buffer only - to be able to dissociate

In [ ]:
fc_buffer_only = ee.FeatureCollection("users/aguerou/ice_and_life/carto_h1b/lia_shp/LIA_Alps_Reinthaler_Paul_2024_Mourey_2025_buffer200_only")

In [ ]:
geemap.ee_export_image_to_drive(carto_g25_with_water_ice.clip(fc_buffer_only).unmask(value=-1), #pas de clip pour que le nodata marche
                                folder="g25_final", 
                                fileNamePrefix="carto_h1b_lia_Reinthaler_Mourey_median_ts_g25_buffer_only", 
                                scale=scale,
                                crs=projection,
                                maxPixels=3e10,
                                region=fc_lia.geometry().convexHull(),
                                formatOptions={"noData": -1})